# Multi-Task Fluorescence Intelligence Network (MTFINet)
## Simultaneous Heavy-Metal Identification and Concentration-Class Prediction
### Revised and Verified Implementation for Journal Submission

**Authors:** Bubryur Kim, Sourasish Nath, Pranav, Sri Preethaa K.R., Yuvaraj Natarajan, Shyamala Devi Munisamy  
**Institution:** Vellore Institute of Technology (VIT), India & Kyungpook National University (KNU), South Korea  

**Key Architecture & Protocol Specifications:**
- **Backbone:** Pretrained ResNet-18 (`torchvision.models.ResNet18_Weights.DEFAULT`)
- **Dual Prediction Heads:**
  1. `metal_head`: 8-class categorical classifier (Copper, Mercury, Palladium, Zinc, Cadmium, Silver, Chromium, Lead)
  2. `conc_head`: 7-class discrete ordinal classifier ($C_0=0, C_1=10, C_2=50, C_3=100, C_4=150, C_5=200, C_6=250\,\mu\text{M}$)
- **Loss Formulation:** Joint Cross-Entropy Loss: $\mathcal{L}_{\text{total}} = \lambda_m \mathcal{L}_m + \lambda_c \mathcal{L}_c$, with $\lambda_m = \lambda_c = 1.0$
- **Optimizer:** Adam with learning rate $\eta = 0.0001$, batch size 16, 25 epochs
- **Evaluation Protocol:** Strictly evaluated on 538 held-out test samples

In [ ]:
import os
import re
from pathlib import Path
from PIL import Image
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    mean_absolute_error,
    cohen_kappa_score
)

# Set compute device
device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
print(f"Active compute device: {device}")

In [ ]:
# --- MultiTaskResNet Model Architecture ---
class MultiTaskResNet(nn.Module):
    def __init__(self, num_metals=8, num_concentrations=7):
        super(MultiTaskResNet, self).__init__()
        self.base_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        num_features = self.base_model.fc.in_features
        self.base_model.fc = nn.Identity() # Remove default ImageNet head
        
        # Task-specific classification heads
        self.metal_head = nn.Linear(num_features, num_metals)
        self.conc_head = nn.Linear(num_features, num_concentrations)
        
    def forward(self, x):
        shared_features = self.base_model(x)
        metal_pred = self.metal_head(shared_features)
        conc_pred = self.conc_head(shared_features)
        return metal_pred, conc_pred

# Initialize and inspect model
model = MultiTaskResNet().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"MTFINet Total Parameters: {total_params:,} ({total_params * 4 / (1024**2):.2f} MB in float32)")

In [ ]:
# --- Dataset Definition & Held-Out Test Evaluation ---
# Specify your local dataset path here if available
dataset_path = Path(os.environ.get('DATASET_DIR', './dataset'))
checkpoint_path = Path('../checkpoints/best_multitask_model_leakage_free.pth')

model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.eval()

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

metals = ['Copper', 'Mercury', 'Palladium', 'Zinc', 'Cadmium', 'Silver', 'Chromium', 'Lead']
concentrations = ['0', '10', '50', '100', '150', '200', '250']
conc_map_uM = np.array([0, 10, 50, 100, 150, 200, 250])
metal_to_idx = {m: i for i, m in enumerate(metals)}
conc_to_idx = {c: i for i, c in enumerate(concentrations)}

# Load strictly the held-out test split (538 images)
test_samples = []
for m in metals:
    t_dir = dataset_path / m / 'Augmented' / 'test'
    if t_dir.exists():
        for c_folder in os.listdir(t_dir):
            c_path = t_dir / c_folder
            if c_path.is_dir():
                match = re.search(r'\d+', c_folder)
                if match and match.group() in conc_to_idx:
                    c_val = match.group()
                    for img_name in os.listdir(c_path):
                        if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                            test_samples.append((c_path / img_name, metal_to_idx[m], conc_to_idx[c_val], int(c_val)))

print(f"Loaded {len(test_samples)} held-out test images.")

In [ ]:
# --- Run Test Predictions and Metric Computation ---
all_m_preds, all_c_preds = [], []
all_m_trues, all_c_trues = [], []

with torch.no_grad():
    for p, m_idx, c_idx, _ in test_samples:
        img = Image.open(p).convert('RGB')
        t_img = eval_transform(img).unsqueeze(0).to(device)
        m_out, c_out = model(t_img)
        all_m_preds.append(torch.argmax(m_out, dim=1).item())
        all_c_preds.append(torch.argmax(c_out, dim=1).item())
        all_m_trues.append(m_idx)
        all_c_trues.append(c_idx)

m_acc = accuracy_score(all_m_trues, all_m_preds)
c_acc = accuracy_score(all_c_trues, all_c_preds)
qer_mae = mean_absolute_error(conc_map_uM[all_c_trues], conc_map_uM[all_c_preds])
qwk = cohen_kappa_score(all_c_trues, all_c_preds, weights='quadratic')

print(f"--- SUMMARY TEST METRICS (N = {len(test_samples)}) ---")
print(f"Metal Classification Accuracy:     {m_acc*100:.2f}% (TEC: {(1-m_acc)*100:.2f}%)")
print(f"Concentration-Class Accuracy:      {c_acc*100:.2f}% (TEC: {(1-c_acc)*100:.2f}%)")
print(f"Mapped-Class MAE (QER):            {qer_mae:.2f} µM")
print(f"Quadratic Weighted Kappa (QWK):    {qwk:.4f}")

print("\n--- Per-Metal Classification Report ---")
print(classification_report(all_m_trues, all_m_preds, target_names=metals, digits=4))

print("\n--- Per-Concentration Classification Report ---")
print(classification_report(all_c_trues, all_c_preds, target_names=[f'{c} µM' for c in concentrations], digits=4))

In [ ]:
# --- Dual Confusion Matrix Visualization ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6), dpi=300)

cm_m = confusion_matrix(all_m_trues, all_m_preds)
sns.heatmap(cm_m, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=metals, yticklabels=metals)
axes[0].set_title(f"(a) Metal Classification on Test Set ({m_acc*100:.2f}%)", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Predicted Metal", fontsize=11)
axes[0].set_ylabel("True Metal", fontsize=11)
axes[0].tick_params(axis='x', rotation=45)

cm_c = confusion_matrix(all_c_trues, all_c_preds)
sns.heatmap(cm_c, annot=True, fmt='d', cmap='Oranges', ax=axes[1],
            xticklabels=[f"{c} µM" for c in concentrations], yticklabels=[f"{c} µM" for c in concentrations])
axes[1].set_title(f"(b) Concentration-Class Prediction on Test Set ({c_acc*100:.2f}%)", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Predicted Concentration", fontsize=11)
axes[1].set_ylabel("True Concentration", fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# --- Physical Sensor Optical Verification: Spectrometer Emission Spectra ---
# Displaying the JASCO FP-8550 research spectrometer data acquired at VIT Vellore
spec_fig = Image.open('../results/figures/jasco_fp8550_emission_spectrum.png')
plt.figure(figsize=(10, 6), dpi=300)
plt.imshow(spec_fig)
plt.axis('off')
plt.title("Physical Benchtop Verification: Carbon-Dot Emission Spectrum (JASCO FP-8550)", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- Image-Derived Optical Quenching Curves with Error Bars ---
# Extracted directly from 2D sensor spots to demonstrate concentration-dependent response
quench_fig = Image.open('../results/figures/image_optical_quenching_curves.png')
plt.figure(figsize=(14, 6), dpi=300)
plt.imshow(quench_fig)
plt.axis('off')
plt.title("Image-Derived Photoluminescence Quenching Response Across Metals and Concentrations", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()